# Elvis Presley Music Analysis - Interactive Visualization

## How to Interact with These Plots:
- **Click and drag** on the scatter plot to select points
- The histogram below will **highlight** songs within your selection
- Songs outside your selection will appear dimmed
- **Click anywhere** on white space to clear your selection

## Time Signature Definition:
**Time signature** indicates the number of beats per measure in a song. Most songs are in 4/4 time (4 beats per measure), but Elvis also recorded songs in 3/4 time (waltz time, like 'Can't Help Falling in Love') and occasionally 5/4 time. The time signature affects the rhythmic feel and structure of the music.

In [ ]:
import pandas as pd
import altair as alt

In [ ]:
# Load Elvis dataset
df = pd.read_csv('dataset.csv')
elvis_df = df[df['artists'] == 'Elvis Presley'].reset_index(drop=True)

# Convert duration
elvis_df['duration_min'] = elvis_df['duration_ms'] / 60000

elvis_df.head()

In [ ]:
# Create a brush selection that will link the two charts
brush = alt.selection_interval(encodings=['x', 'y'])

In [ ]:
# Define green color scale for valence
green_scale = alt.Scale(
    domain=[0, 1],
    range=['#355E3B', '#90EE90']  # Hunter Green to Light Green
)

# Scatter plot with brush selection
scatter = (
    alt.Chart(elvis_df)
    .mark_circle(size=100, opacity=0.7)
    .encode(
        x=alt.X('danceability:Q', title='Danceability'),
        y=alt.Y('energy:Q', title='Energy'),
        color=alt.condition(
            brush,
            alt.Color('valence:Q', scale=green_scale, title='Valence (Mood)'),
            alt.value('#CCCCCC')  # Gray for unselected points
        ),
        size=alt.Size('popularity:Q', title='Popularity'),
        tooltip=['track_name', 'album_name', 'popularity', 'duration_min', 'valence', 'tempo', 'time_signature']
    )
    .add_params(brush)
    .properties(
        title='Elvis Presley Songs: Energy vs Danceability (Click and drag to select)',
        width=600,
        height=400
    )
)

In [ ]:
# Histogram linked to scatter plot selection
# Base histogram showing all data in gray
duration_hist_base = (
    alt.Chart(elvis_df)
    .mark_bar(opacity=0.3)
    .encode(
        x=alt.X('duration_min:Q', bin=alt.Bin(maxbins=20), title='Song Duration (minutes)'),
        y=alt.Y('count()', title='Number of Songs'),
        color=alt.value('#CCCCCC')  # Gray for all data
    )
)

# Overlay histogram showing only selected data
duration_hist_selected = (
    alt.Chart(elvis_df)
    .mark_bar(opacity=0.8)
    .encode(
        x=alt.X('duration_min:Q', bin=alt.Bin(maxbins=20), title='Song Duration (minutes)'),
        y=alt.Y('count()', title='Number of Songs'),
        color=alt.value('#2AAA8A')  # Jungle Green for selected
    )
    .transform_filter(brush)
)

# Layer the histograms
duration_hist = (duration_hist_base + duration_hist_selected).properties(
    title='Distribution of Song Durations (Highlights selected songs from above)',
    width=600,
    height=300
)

In [ ]:
# Combine charts vertically
final_chart = scatter & duration_hist

final_chart

In [ ]:
# Save to HTML
final_chart.save('ep_comparison_interactive.html')